In [ ]:
!pip install emoji pyarabic nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 9.0 MB/s eta 0:00:00


In [ ]:
## Preprocessing and Cleaning:
import nltk
import emoji
import pandas as pd
import pyarabic.araby as araby
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from nltk.tokenize import word_tokenize
import re

## Text Representation:
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

#**1- Load Dataset:**

In [ ]:
df=pd.read_csv("AAFAQ_Dataset.csv")

df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.


#**2- Preprocessing:**



In [ ]:
##Check if the dataset contains Emoji:

num_emoji=df['QuestionText'].apply(
    lambda x: any(char in emoji.EMOJI_DATA for char in x)).sum()
num_emoji2=df['Answer'].apply(
    lambda x: any(char in emoji.EMOJI_DATA for char in x)).sum()
print(num_emoji)
print(num_emoji2)

## ---> So no need for emoji removing

0
0


In [ ]:
## Remove Tashkeel:

def remove_diacritics(text):
    return araby.strip_tashkeel(text)

df['QuestionText']=df['QuestionText'].apply(remove_diacritics)
df['Answer']=df['Answer'].apply(remove_diacritics)
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساسا من النيتروجين؟,التعليم,الهواء يتكون أساسا من النيتروجين.


In [ ]:
## Remove Tatweel:

def remove_tatweel(text):
    return araby.strip_tatweel(text)

df['QuestionText']=df['QuestionText'].apply(remove_tatweel)
df['Answer']=df['Answer'].apply(remove_tatweel)
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساسا من النيتروجين؟,التعليم,الهواء يتكون أساسا من النيتروجين.


In [ ]:
## Removing StopWords:
stop_words= set(stopwords.words('arabic'))

def removeStopwords(sentence):
  words = sentence.split()
  filtered = [w for w in words if w not in stop_words]
  return " ".join(filtered)

df['QuestionText']=df['QuestionText'].apply(removeStopwords)
df['Answer']=df['Answer'].apply(removeStopwords)
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة السابق الوقت الحالي؟,التعليم,الدراسة الوقت الحالي تعتبر أفضل بسبب توفر التك...
1,أليس القطن عماد الثروة مصر؟,الاقتصاد والعمل,القطن يعتبر أهم المنتجات الزراعية مصر، ويعد ال...
2,أتصعد الشمس الشرق؟,التعليم,الشمس تصعد الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساسا النيتروجين؟,التعليم,الهواء يتكون أساسا النيتروجين.


In [ ]:
## Normalize Hamza:

def normalize_hamza(text):
  text = re.sub("[إأٱآا]", "ا", text)
  text = re.sub(r'ؤ', 'و', text)
  text = re.sub(r'ئ', 'ي', text)
  text = re.sub(r'ة', 'ه', text)
  text = re.sub(r'ء', '', text)
  return text

df['QuestionText']=df['QuestionText'].apply(normalize_hamza)
df['Answer']=df['Answer'].apply(normalize_hamza)
df.head()

,QuestionText,Category,Answer
0,ايهما افضل الدراسه السابق الوقت الحالي؟,التعليم,الدراسه الوقت الحالي تعتبر افضل بسبب توفر التك...
1,اليس القطن عماد الثروه مصر؟,الاقتصاد والعمل,القطن يعتبر اهم المنتجات الزراعيه مصر، ويعد ال...
2,اتصعد الشمس الشرق؟,التعليم,الشمس تصعد الشرق.
3,اتعرف البكتيريا بانها كاينات حيه دقيقه؟,التعليم,البكتيريا تعرف بانها كاينات حيه دقيقه.
4,ايتكون الهوا اساسا النيتروجين؟,التعليم,الهوا يتكون اساسا النيتروجين.


In [ ]:
## Tokanization:

df['QuestionText']=df['QuestionText'].apply(araby.tokenize)
df['Answer']=df['Answer'].apply(araby.tokenize)
df.head()

,QuestionText,Category,Answer
0,"[ايهما, افضل, الدراسه, السابق, الوقت, الحالي, ؟]",التعليم,"[الدراسه, الوقت, الحالي, تعتبر, افضل, بسبب, تو..."
1,"[اليس, القطن, عماد, الثروه, مصر, ؟]",الاقتصاد والعمل,"[القطن, يعتبر, اهم, المنتجات, الزراعيه, مصر, ،..."
2,"[اتصعد, الشمس, الشرق, ؟]",التعليم,"[الشمس, تصعد, الشرق, .]"
3,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه, ؟]",التعليم,"[البكتيريا, تعرف, بانها, كاينات, حيه, دقيقه, .]"
4,"[ايتكون, الهوا, اساسا, النيتروجين, ؟]",التعليم,"[الهوا, يتكون, اساسا, النيتروجين, .]"


In [ ]:
## Removing Punctioation:

def remove_punctiation(text_tokens):
  return [token for token in text_tokens if token.isalnum()]

df['QuestionText']=df['QuestionText'].apply(remove_punctiation)
df['Answer']=df['Answer'].apply(remove_punctiation)
df.head()

,QuestionText,Category,Answer
0,"[ايهما, افضل, الدراسه, السابق, الوقت, الحالي]",التعليم,"[الدراسه, الوقت, الحالي, تعتبر, افضل, بسبب, تو..."
1,"[اليس, القطن, عماد, الثروه, مصر]",الاقتصاد والعمل,"[القطن, يعتبر, اهم, المنتجات, الزراعيه, مصر, و..."
2,"[اتصعد, الشمس, الشرق]",التعليم,"[الشمس, تصعد, الشرق]"
3,"[اتعرف, البكتيريا, بانها, كاينات, حيه, دقيقه]",التعليم,"[البكتيريا, تعرف, بانها, كاينات, حيه, دقيقه]"
4,"[ايتكون, الهوا, اساسا, النيتروجين]",التعليم,"[الهوا, يتكون, اساسا, النيتروجين]"


In [ ]:
## Stemming:

stemmer=ISRIStemmer()
def steeming(text_tokens):
  return [stemmer.stem(token) for token in text_tokens]

df['QuestionText']=df['QuestionText'].apply(steeming)
df['Answer']=df['Answer'].apply(steeming)
df.head()

,QuestionText,Category,Answer
0,"[ايه, فضل, درس, سبق, وقت, الحالي]",التعليم,"[درس, وقت, الحالي, عبر, فضل, سبب, وفر, كنولوج,..."
1,"[الس, قطن, عمد, ثره, مصر]",الاقتصاد والعمل,"[قطن, عبر, اهم, نتج, زرع, مصر, يعد, عمد, ريس, ..."
2,"[صعد, شمس, شرق]",التعليم,"[شمس, صعد, شرق]"
3,"[عرف, كتر, بان, كين, حيه, دقق]",التعليم,"[كتر, عرف, بان, كين, حيه, دقق]"
4,"[ايت, هوا, سسا, ترج]",التعليم,"[هوا, يتك, سسا, ترج]"


In [ ]:
df['QuestionText']=df['QuestionText'].apply(lambda tokens: ' '.join(tokens))
df['Answer']=df['Answer'].apply(lambda tokens: ' '.join(tokens))

# Save processed dataset
df.to_csv('AAFAQ_Final_Processed.csv', index=False, encoding='utf-8-sig')

print("Saved: AAFAQ_Processed.csv")
df.head()

Saved: AAFAQ_Processed.csv


,QuestionText,Category,Answer
0,ايه فضل درس سبق وقت الحالي,التعليم,درس وقت الحالي عبر فضل سبب وفر كنولوج ورد علم حدث
1,الس قطن عمد ثره مصر,الاقتصاد والعمل,قطن عبر اهم نتج زرع مصر يعد عمد ريس قصد صري
2,صعد شمس شرق,التعليم,شمس صعد شرق
3,عرف كتر بان كين حيه دقق,التعليم,كتر عرف بان كين حيه دقق
4,ايت هوا سسا ترج,التعليم,هوا يتك سسا ترج
